# MUB / VTEB Deep Dive

Starter notebook for inspecting price history and existing scan results for one ETF pair.

In [ ]:
import importlib.util
import sys

print(sys.executable)
for package in ["pandas", "matplotlib", "yfinance", "statsmodels"]:
    spec = importlib.util.find_spec(package)
    print(package, "found" if spec else "missing", spec.origin if spec else "")

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

TICKER_A = "MUB"
TICKER_B = "VTEB"

cwd = Path.cwd()
if cwd.name == "notebooks":
    project_dir = cwd.parent
elif (cwd / "pairs_trading").exists():
    project_dir = cwd / "pairs_trading"
else:
    project_dir = cwd

data_dir = project_dir / "data"
outputs_dir = project_dir / "outputs"

price_history_path = data_dir / "etf_price_history.csv"
scan_path = outputs_dir / "cointegration_scan.csv"
rolling_summary_path = outputs_dir / f"rolling_{TICKER_A}_{TICKER_B}_summary.csv"
viability_path = outputs_dir / f"viability_{TICKER_A}_{TICKER_B}.csv"
rolling_windows_path = outputs_dir / f"rolling_{TICKER_A}_{TICKER_B}.csv"

price_history_path, scan_path, rolling_windows_path, rolling_summary_path, viability_path

## Price History

In [ ]:
prices = pd.read_csv(price_history_path, parse_dates=["date"])
pair_prices = (
    prices[prices["ticker"].isin([TICKER_A, TICKER_B])]
    .pivot(index="date", columns="ticker", values="adj_close")
    .dropna()
    .sort_index()
)

pair_prices.tail()

In [ ]:
normalized = pair_prices / pair_prices.iloc[0] * 100

ax = normalized.plot(figsize=(11, 5), linewidth=1.5)
ax.set_title(f"{TICKER_A} / {TICKER_B} normalized adjusted close")
ax.set_ylabel("Indexed value, first date = 100")
ax.grid(True, alpha=0.25)
plt.show()

## Scan Results

In [ ]:
scan = pd.read_csv(scan_path)
pair_scan = scan[
    ((scan["ticker_a"] == TICKER_A) & (scan["ticker_b"] == TICKER_B))
    | ((scan["ticker_a"] == TICKER_B) & (scan["ticker_b"] == TICKER_A))
]

pair_scan.T

## Spread

In [ ]:
scan_row = pair_scan.iloc[0]
hedge_ratio = scan_row["hedge_ratio"]
intercept = scan_row["intercept"]

log_prices = np.log(pair_prices[[TICKER_A, TICKER_B]])
spread = log_prices[TICKER_A] - intercept - hedge_ratio * log_prices[TICKER_B]

spread_summary = pd.Series(
    {
        "hedge_ratio": hedge_ratio,
        "intercept": intercept,
        "spread_mean": spread.mean(),
        "spread_std": spread.std(),
        "spread_std_bps": spread.std() * 10_000,
    }
)
spread_summary

In [ ]:
ax = spread.plot(figsize=(11, 5), linewidth=1.2)
ax.axhline(spread.mean(), color="black", linewidth=1, linestyle="--", label="mean")
ax.axhline(spread.mean() + spread.std(), color="gray", linewidth=1, linestyle=":", label="+/-1 std")
ax.axhline(spread.mean() - spread.std(), color="gray", linewidth=1, linestyle=":")
ax.set_title(f"{TICKER_A} / {TICKER_B} hedge-ratio spread")
ax.set_ylabel("Log spread")
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()

## Z-Score

In [ ]:
zscore = (spread - spread.mean()) / spread.std()

ax = zscore.plot(figsize=(11, 5), linewidth=1.2)
for level, color, linestyle in [
    (2.0, "crimson", "--"),
    (-2.0, "crimson", "--"),
    (0.5, "darkgreen", ":"),
    (-0.5, "darkgreen", ":"),
    (0.0, "black", "-"),
]:
    ax.axhline(level, color=color, linewidth=1, linestyle=linestyle)

ax.set_title(f"{TICKER_A} / {TICKER_B} spread z-score")
ax.set_ylabel("Z-score")
ax.grid(True, alpha=0.25)
plt.show()

## Entry / Exit Signals

In [ ]:
entry_z = 2.0
exit_z = 0.5
position = 0
signals = []

for date, z in zscore.items():
    if position == 0:
        if z >= entry_z:
            position = -1
            signals.append({"date": date, "signal": "short_spread_entry", "zscore": z})
        elif z <= -entry_z:
            position = 1
            signals.append({"date": date, "signal": "long_spread_entry", "zscore": z})
    elif position == 1 and z >= -exit_z:
        signals.append({"date": date, "signal": "long_spread_exit", "zscore": z})
        position = 0
    elif position == -1 and z <= exit_z:
        signals.append({"date": date, "signal": "short_spread_exit", "zscore": z})
        position = 0

signals = pd.DataFrame(signals)
signals.tail(20)

In [ ]:
ax = zscore.plot(figsize=(11, 5), linewidth=1.1)
for level, color, linestyle in [
    (entry_z, "crimson", "--"),
    (-entry_z, "crimson", "--"),
    (exit_z, "darkgreen", ":"),
    (-exit_z, "darkgreen", ":"),
    (0.0, "black", "-"),
]:
    ax.axhline(level, color=color, linewidth=1, linestyle=linestyle)

if not signals.empty:
    entries = signals[signals["signal"].str.endswith("entry")]
    exits = signals[signals["signal"].str.endswith("exit")]
    ax.scatter(entries["date"], entries["zscore"], color="crimson", marker="^", s=45, label="entries")
    ax.scatter(exits["date"], exits["zscore"], color="darkgreen", marker="v", s=45, label="exits")

ax.set_title(f"{TICKER_A} / {TICKER_B} z-score with entry/exit signals")
ax.set_ylabel("Z-score")
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()

## Trade List

In [ ]:
cost_bps = 4.0
position = 0
open_trade = None
trades = []

for date, z in zscore.items():
    current_spread = spread.loc[date]
    if position == 0:
        if z >= entry_z:
            position = -1
            open_trade = {
                "entry_date": date,
                "direction": "short_spread",
                "entry_z": z,
                "entry_spread": current_spread,
            }
        elif z <= -entry_z:
            position = 1
            open_trade = {
                "entry_date": date,
                "direction": "long_spread",
                "entry_z": z,
                "entry_spread": current_spread,
            }
    elif position == 1 and z >= -exit_z:
        gross_pnl_bps = (current_spread - open_trade["entry_spread"]) * 10_000
        trades.append({
            **open_trade,
            "exit_date": date,
            "exit_z": z,
            "exit_spread": current_spread,
            "days_held": (date - open_trade["entry_date"]).days,
            "gross_pnl_bps": gross_pnl_bps,
            "cost_bps": cost_bps,
            "net_pnl_bps": gross_pnl_bps - cost_bps,
        })
        position = 0
        open_trade = None
    elif position == -1 and z <= exit_z:
        gross_pnl_bps = (open_trade["entry_spread"] - current_spread) * 10_000
        trades.append({
            **open_trade,
            "exit_date": date,
            "exit_z": z,
            "exit_spread": current_spread,
            "days_held": (date - open_trade["entry_date"]).days,
            "gross_pnl_bps": gross_pnl_bps,
            "cost_bps": cost_bps,
            "net_pnl_bps": gross_pnl_bps - cost_bps,
        })
        position = 0
        open_trade = None

trades = pd.DataFrame(trades)
trades.tail(10)

In [ ]:
trade_summary = pd.Series(
    {
        "completed_trades": len(trades),
        "win_rate": (trades["net_pnl_bps"] > 0).mean() if not trades.empty else float("nan"),
        "avg_days_held": trades["days_held"].mean() if not trades.empty else float("nan"),
        "median_days_held": trades["days_held"].median() if not trades.empty else float("nan"),
        "avg_gross_pnl_bps": trades["gross_pnl_bps"].mean() if not trades.empty else float("nan"),
        "avg_net_pnl_bps": trades["net_pnl_bps"].mean() if not trades.empty else float("nan"),
        "total_net_pnl_bps": trades["net_pnl_bps"].sum() if not trades.empty else 0.0,
        "worst_net_pnl_bps": trades["net_pnl_bps"].min() if not trades.empty else float("nan"),
        "best_net_pnl_bps": trades["net_pnl_bps"].max() if not trades.empty else float("nan"),
    }
)
trade_summary

## Walk-Forward Trade Prototype

In [ ]:
import statsmodels.api as sm

formation_days = 504
wf_rows = []

for idx in range(formation_days, len(pair_prices)):
    formation_prices = pair_prices.iloc[idx - formation_days : idx]
    current_date = pair_prices.index[idx]
    current_prices = pair_prices.iloc[idx]

    formation_log = np.log(formation_prices[[TICKER_A, TICKER_B]])
    model = sm.OLS(
        formation_log[TICKER_A],
        sm.add_constant(formation_log[TICKER_B]),
    ).fit()
    wf_intercept = model.params["const"]
    wf_hedge_ratio = model.params[TICKER_B]
    formation_spread = (
        formation_log[TICKER_A]
        - wf_intercept
        - wf_hedge_ratio * formation_log[TICKER_B]
    )
    current_spread = (
        np.log(current_prices[TICKER_A])
        - wf_intercept
        - wf_hedge_ratio * np.log(current_prices[TICKER_B])
    )
    wf_zscore = (current_spread - formation_spread.mean()) / formation_spread.std()

    wf_rows.append(
        {
            "date": current_date,
            "spread": current_spread,
            "zscore": wf_zscore,
            "hedge_ratio": wf_hedge_ratio,
            "intercept": wf_intercept,
            "formation_spread_mean": formation_spread.mean(),
            "formation_spread_std": formation_spread.std(),
        }
    )

walk_forward = pd.DataFrame(wf_rows).set_index("date")
walk_forward.tail()

In [ ]:
ax = walk_forward["zscore"].plot(figsize=(11, 5), linewidth=1.1)
for level, color, linestyle in [
    (entry_z, "crimson", "--"),
    (-entry_z, "crimson", "--"),
    (exit_z, "darkgreen", ":"),
    (-exit_z, "darkgreen", ":"),
    (0.0, "black", "-"),
]:
    ax.axhline(level, color=color, linewidth=1, linestyle=linestyle)

ax.set_title(f"{TICKER_A} / {TICKER_B} walk-forward z-score")
ax.set_ylabel("Z-score")
ax.grid(True, alpha=0.25)
plt.show()

In [ ]:
position = 0
open_trade = None
wf_trades = []

for date, row in walk_forward.iterrows():
    z = row["zscore"]
    current_spread = row["spread"]
    if position == 0:
        if z >= entry_z:
            position = -1
            open_trade = {
                "entry_date": date,
                "direction": "short_spread",
                "entry_z": z,
                "entry_spread": current_spread,
                "entry_hedge_ratio": row["hedge_ratio"],
            }
        elif z <= -entry_z:
            position = 1
            open_trade = {
                "entry_date": date,
                "direction": "long_spread",
                "entry_z": z,
                "entry_spread": current_spread,
                "entry_hedge_ratio": row["hedge_ratio"],
            }
    elif position == 1 and z >= -exit_z:
        gross_pnl_bps = (current_spread - open_trade["entry_spread"]) * 10_000
        wf_trades.append({
            **open_trade,
            "exit_date": date,
            "exit_z": z,
            "exit_spread": current_spread,
            "days_held": (date - open_trade["entry_date"]).days,
            "gross_pnl_bps": gross_pnl_bps,
            "cost_bps": cost_bps,
            "net_pnl_bps": gross_pnl_bps - cost_bps,
        })
        position = 0
        open_trade = None
    elif position == -1 and z <= exit_z:
        gross_pnl_bps = (open_trade["entry_spread"] - current_spread) * 10_000
        wf_trades.append({
            **open_trade,
            "exit_date": date,
            "exit_z": z,
            "exit_spread": current_spread,
            "days_held": (date - open_trade["entry_date"]).days,
            "gross_pnl_bps": gross_pnl_bps,
            "cost_bps": cost_bps,
            "net_pnl_bps": gross_pnl_bps - cost_bps,
        })
        position = 0
        open_trade = None

wf_trades = pd.DataFrame(wf_trades)
wf_trades.tail(10)

In [ ]:
wf_trade_summary = pd.Series(
    {
        "completed_trades": len(wf_trades),
        "open_position": position,
        "win_rate": (wf_trades["net_pnl_bps"] > 0).mean() if not wf_trades.empty else float("nan"),
        "avg_days_held": wf_trades["days_held"].mean() if not wf_trades.empty else float("nan"),
        "median_days_held": wf_trades["days_held"].median() if not wf_trades.empty else float("nan"),
        "avg_gross_pnl_bps": wf_trades["gross_pnl_bps"].mean() if not wf_trades.empty else float("nan"),
        "avg_net_pnl_bps": wf_trades["net_pnl_bps"].mean() if not wf_trades.empty else float("nan"),
        "total_net_pnl_bps": wf_trades["net_pnl_bps"].sum() if not wf_trades.empty else 0.0,
        "worst_net_pnl_bps": wf_trades["net_pnl_bps"].min() if not wf_trades.empty else float("nan"),
        "best_net_pnl_bps": wf_trades["net_pnl_bps"].max() if not wf_trades.empty else float("nan"),
    }
)
wf_trade_summary

## Cost Sensitivity

In [ ]:
cost_levels_bps = [0, 2, 4, 8, 12, 16, 20, 24, 28, 32]
cost_sensitivity_rows = []

for test_cost_bps in cost_levels_bps:
    net_pnl = wf_trades["gross_pnl_bps"] - test_cost_bps
    cost_sensitivity_rows.append(
        {
            "round_trip_cost_bps": test_cost_bps,
            "completed_trades": len(wf_trades),
            "win_rate": (net_pnl > 0).mean(),
            "avg_net_pnl_bps": net_pnl.mean(),
            "median_net_pnl_bps": net_pnl.median(),
            "total_net_pnl_bps": net_pnl.sum(),
            "worst_net_pnl_bps": net_pnl.min(),
            "best_net_pnl_bps": net_pnl.max(),
        }
    )

cost_sensitivity = pd.DataFrame(cost_sensitivity_rows)
break_even_cost_bps = wf_trades["gross_pnl_bps"].mean() if not wf_trades.empty else float("nan")
cost_sensitivity

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

axes[0].plot(cost_sensitivity["round_trip_cost_bps"], cost_sensitivity["total_net_pnl_bps"], marker="o")
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_ylabel("Total net P&L, bps")
axes[0].set_title(f"{TICKER_A} / {TICKER_B} walk-forward cost sensitivity")

axes[1].plot(cost_sensitivity["round_trip_cost_bps"], cost_sensitivity["win_rate"], marker="o")
axes[1].set_ylabel("Win rate")
axes[1].set_xlabel("Round-trip cost, bps")
axes[1].set_ylim(0, 1.05)

for ax in axes:
    ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

break_even_cost_bps

In [ ]:
rolling_summary = pd.read_csv(rolling_summary_path)
rolling_summary.T

## Rolling Diagnostics

In [ ]:
rolling = pd.read_csv(rolling_windows_path, parse_dates=["start_date", "end_date"])
rolling.tail()

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(11, 11), sharex=True)

axes[0].plot(rolling["end_date"], rolling["coint_pvalue"], linewidth=1.2)
axes[0].axhline(0.05, color="crimson", linestyle="--", linewidth=1)
axes[0].set_ylabel("Coint p")
axes[0].set_title(f"{TICKER_A} / {TICKER_B} rolling diagnostics")

axes[1].plot(rolling["end_date"], rolling["adf_pvalue"], linewidth=1.2)
axes[1].axhline(0.05, color="crimson", linestyle="--", linewidth=1)
axes[1].set_ylabel("ADF p")

axes[2].plot(rolling["end_date"], rolling["hedge_ratio"], linewidth=1.2)
axes[2].axhline(rolling["hedge_ratio"].median(), color="black", linestyle=":", linewidth=1)
axes[2].set_ylabel("Hedge ratio")

axes[3].plot(rolling["end_date"], rolling["half_life_days"], linewidth=1.2)
axes[3].axhline(1, color="crimson", linestyle="--", linewidth=1)
axes[3].axhline(45, color="crimson", linestyle="--", linewidth=1)
axes[3].set_ylabel("Half-life")

for ax in axes:
    ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
viability = pd.read_csv(viability_path)
viability.T